In [88]:
## Import libraries (numpy, pandas, ...)
import pandas as pd
import numpy as np
from typing import Dict
import statsmodels.api as sm

In [89]:
## Import Divvy trip data
divvy_trips = pd.read_stata("data/divvy_data.dta")
divvy_trips

,start_date,from_station_id,trips
0,2013-06-27,17,4.0
1,2013-06-27,19,2.0
2,2013-06-27,20,1.0
3,2013-06-27,24,1.0
4,2013-06-27,28,1.0
...,...,...,...
951667,2019-12-31,659,3.0
951668,2019-12-31,660,2.0
951669,2019-12-31,664,1.0
951670,2019-12-31,672,21.0


In [90]:
## Import Divvy location dataset
divvy_locations = pd.read_stata("data/IDlatlong.dta")
divvy_locations

,from_station_id,Latitude,Longitude
0,2.0,41.876511,-87.620548
1,3.0,41.867226,-87.615355
2,4.0,41.856268,-87.613348
3,5.0,41.874053,-87.627716
4,6.0,41.886976,-87.612813
...,...,...,...
604,664.0,41.939354,-87.683282
605,665.0,41.747363,-87.580046
606,666.0,41.907221,-87.655618
607,672.0,41.891023,-87.635480


In [91]:
## Merge Datasets
divvy = pd.merge(divvy_trips, divvy_locations, left_on="from_station_id", right_on="from_station_id", how="left")
divvy

,start_date,from_station_id,trips,Latitude,Longitude
0,2013-06-27,17,4.0,41.903119,-87.673935
1,2013-06-27,19,2.0,41.868968,-87.659141
2,2013-06-27,20,1.0,41.910522,-87.653106
3,2013-06-27,24,1.0,41.891847,-87.620580
4,2013-06-27,28,1.0,41.914680,-87.643320
...,...,...,...,...,...
951667,2019-12-31,659,3.0,41.895501,-87.682017
951668,2019-12-31,660,2.0,42.004583,-87.661406
951669,2019-12-31,664,1.0,41.939354,-87.683282
951670,2019-12-31,672,21.0,41.891023,-87.635480


In [92]:
# Haversine distance in miles
def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth radius in miles

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c



def div_filter( divvy : pd.DataFrame, c : tuple[float], r :float):
###
# dataset filtering function; takes center, radius info for filtering
# params: 
#   divvy (DataFrame): DataFrame w/ divvy bike ride info including longitude and latitude values for station, station ID, and date (YYY-MM-DD)
#   c (tuple(float)) : (c is short for "center"), this is the center longitude and latitude tuple defining the midpoint of the area for which the divvy data will be filtered for distance 
#   r (float) : radius (in Mi), the radius defining the circular boundary around the center ('c') point (divvy ride entried within this radius will be added to the output dataframe)
# outputs: 
#   inrange (DataFrame) : a dataframe filtered for divvy bikle rides within the specified radius around the provided center point
###
  # coordinates lat, long sep into single vars for computations
  lat = c[0]
  lon = c[1]

  # Calculate distance from each station to Soldier Field
  divvy["dist_to_c_mi"] = haversine_miles(
      lat,
      lon,
      divvy["Latitude"],
      divvy["Longitude"]
  )

  inrange = divvy[divvy["dist_to_c_mi"] <= r].copy()
  return inrange
  

In [93]:
def split_df_on_game_days(divvy: pd.DataFrame, gds_by_yr: dict):
    all_gds = set()
    for dates in gds_by_yr.values():
        all_gds |= dates

    # Build a mask: within any season's first-to-last game range
    in_season = pd.Series(False, index=divvy.index)
    for yr_dates in gds_by_yr.values():
        season_start = min(yr_dates)
        season_end   = max(yr_dates)
        in_season |= (divvy["start_date"] >= season_start) & (divvy["start_date"] <= season_end)

    game_day_mask = divvy["start_date"].isin(all_gds)

    gd    = divvy[game_day_mask].copy(); print("GAME DAY"); print(gd)
    notgd = divvy[in_season & ~game_day_mask].copy(); print("\033[1m" +"*NOT* " + "\033[0m" + "GAME DAY"); print(notgd)

    return gd, notgd


In [94]:
soldiers_coords = (41.8625, -87.6167)
# returns df of all divvy ride entries within 1 mi of soldiers field center pt as defined above
soldiers_df = div_filter(divvy, soldiers_coords, r=1.0) 

jackson_coords = (41.7831, -87.5819)
jackson_df = div_filter(divvy, jackson_coords, r=1.0) # does the same but now for jackson park (control area)



#now need to create a dict to hold the game days, will use the years as keys then use a set to hold the dates in same format as the divvy df does ('YYY-MM-DD' string)
gds_by_yr = {
    '2017': {
        pd.Timestamp('2017-09-10'),
        pd.Timestamp('2017-09-24'),
        pd.Timestamp('2017-10-09'),
        pd.Timestamp('2017-10-22'),
        pd.Timestamp('2017-11-12'),
        pd.Timestamp('2017-11-19'),
        pd.Timestamp('2017-12-03'),
    },
    '2018': {
        pd.Timestamp('2018-09-17'),
        pd.Timestamp('2018-09-30'),
        pd.Timestamp('2018-10-21'),
        pd.Timestamp('2018-10-28'),
        pd.Timestamp('2018-11-11'),
        pd.Timestamp('2018-11-18'),
        pd.Timestamp('2018-12-09'),
        pd.Timestamp('2018-12-16'),
    }
}

# now need to separate based on game-day or non-game day entries
s_gd, s_notgd = split_df_on_game_days(soldiers_df, gds_by_yr)
j_gd, j_notgd = split_df_on_game_days(jackson_df, gds_by_yr)


GAME DAY
       start_date  from_station_id  trips   Latitude  Longitude  dist_to_c_mi
553997 2017-09-10                2  101.0  41.876511 -87.620548      0.988131
553998 2017-09-10                3  247.0  41.867226 -87.615355      0.333785
553999 2017-09-10                4  130.0  41.856268 -87.613348      0.463860
554000 2017-09-10                5   23.0  41.874053 -87.627716      0.979012
554034 2017-09-10               41   37.0  41.872078 -87.629544      0.935233
...           ...              ...    ...        ...        ...           ...
768771 2018-12-16              341   18.0  41.866095 -87.607267      0.545252
768786 2018-12-16              370    2.0  41.854184 -87.619154      0.588281
768796 2018-12-16              394    7.0  41.870816 -87.631246      0.943576
768894 2018-12-16              623   16.0  41.872773 -87.623981      0.802603
768897 2018-12-16              626   11.0  41.867491 -87.632190      0.868451

[313 rows x 6 columns]
*NOT* GAME DAY
       start_dat

In [95]:
#now with the 4 separate datasets (for A,B,C,D analagously in DiD table),
# can do the DiD calcualtion using th statsmodels api....

s_gd["treated"] = 1; s_gd["game_day"] = 1
s_notgd["treated"] = 1; s_notgd["game_day"] = 0
j_gd["treated"] = 0; j_gd["game_day"] = 1
j_notgd["treated"] = 0; j_notgd["game_day"] = 0

combined = pd.concat ([s_gd, s_notgd, j_gd, j_notgd], ignore_index=True)
daily = (combined.groupby(["start_date", "treated", "game_day"])["trips"].sum().reset_index())

daily["DiD"] = daily["treated"] * daily["game_day"]

#now can run OLS
X = sm.add_constant(daily[["treated", "game_day", "DiD"]])
y = daily["trips"]

model = sm.OLS(y, X).fit()
print(model.summary())



                            OLS Regression Results                            
Dep. Variable:                  trips   R-squared:                       0.485
Model:                            OLS   Adj. R-squared:                  0.480
Method:                 Least Squares   F-statistic:                     109.1
Date:                Tue, 19 May 2026   Prob (F-statistic):           8.13e-50
Time:                        00:35:59   Log-Likelihood:                -2430.3
No. Observations:                 352   AIC:                             4869.
Df Residuals:                     348   BIC:                             4884.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         77.8944     19.110      4.076      0.0

In [96]:
# ============================================================
# Setup
# ============================================================
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
import warnings
warnings.filterwarnings("ignore")

def add_time_vars(df):
    df = df.copy()
    df["start_date"] = pd.to_datetime(df["start_date"])
    df["DiD"] = df["treated"] * df["game_day"]
    df["dow"] = df["start_date"].dt.dayofweek
    df["month"] = df["start_date"].dt.month
    df["year"] = df["start_date"].dt.year
    return df

def run_three_specs(df, title="DiD Results"):
    m1 = smf.ols(
        "trips ~ treated + game_day + DiD",
        data=df
    ).fit(cov_type="HC3")

    m2 = smf.ols(
        "trips ~ treated + game_day + DiD + C(dow) + C(month)",
        data=df
    ).fit(cov_type="HC3")

    m3 = smf.ols(
        "trips ~ game_day + DiD + C(from_station_id) + C(dow) + C(month)",
        data=df
    ).fit(cov_type="HC3")

    table = summary_col(
        [m1, m2, m3],
        model_names=["M1 Basic", "M2 Time FE", "M3 Station FE"],
        stars=True,
        float_format="%0.3f",
        info_dict={
            "N": lambda x: f"{int(x.nobs)}",
            "R2": lambda x: f"{x.rsquared:.3f}"
        },
        regressor_order=["DiD", "treated", "game_day"]
    )

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    print(table)
    return m1, m2, m3

In [97]:
# ============================================================
# 1. Fixed Effects DiD — Main Result with Jackson Park Control
# ============================================================

combined_fe = add_time_vars(combined)

df = combined_fe[combined_fe["year"].isin([2017, 2018])].copy()

print(f"Study window rows : {len(df):,}")
print(f"Game-day rows     : {df['game_day'].sum():,}")
print(f"Treated rows      : {df['treated'].sum():,}")
print(f"DiD rows          : {df['DiD'].sum():,}")

m1, m2, m3 = run_three_specs(
    df,
    title="Main DiD: Soldier Field vs. Jackson Park, 2017–2018"
)

Study window rows : 5,045
Game-day rows     : 426
Treated rows      : 3,677
DiD rows          : 313

Main DiD: Soldier Field vs. Jackson Park, 2017–2018

                           M1 Basic M2 Time FE M3 Station FE
------------------------------------------------------------
DiD                       6.868***  7.653***   7.742***     
                          (2.584)   (2.506)    (2.037)      
treated                   15.371*** 16.568***               
                          (0.556)   (0.580)                 
game_day                  -0.072    -1.051     -1.027       
                          (1.149)   (1.815)    (1.473)      
Intercept                 9.993***  24.365***  38.100***    
                          (0.311)   (1.163)    (2.099)      
C(dow)[T.1]                         -2.330**   -2.505***    
                                    (1.046)    (0.792)      
C(dow)[T.2]                         -2.476**   -2.585***    
                                    (1.020)    (0.763

In [98]:
# ============================================================
# 2. Humboldt Park Robustness Check
# ============================================================

humboldt_coords = (41.9000, -87.7200)

humboldt_df = div_filter(divvy, humboldt_coords, r=1.0)

h_gd, h_notgd = split_df_on_game_days(humboldt_df, gds_by_yr)

s_gd_h = s_gd.copy()
s_notgd_h = s_notgd.copy()
h_gd = h_gd.copy()
h_notgd = h_notgd.copy()

s_gd_h["treated"] = 1
s_gd_h["game_day"] = 1

s_notgd_h["treated"] = 1
s_notgd_h["game_day"] = 0

h_gd["treated"] = 0
h_gd["game_day"] = 1

h_notgd["treated"] = 0
h_notgd["game_day"] = 0

humboldt_combined = pd.concat(
    [s_gd_h, s_notgd_h, h_gd, h_notgd],
    ignore_index=True
)

humboldt_combined = add_time_vars(humboldt_combined)
humboldt_df_1718 = humboldt_combined[
    humboldt_combined["year"].isin([2017, 2018])
].copy()

hm1, hm2, hm3 = run_three_specs(
    humboldt_df_1718,
    title="Robustness: Soldier Field vs. Humboldt Park, 2017–2018"
)

GAME DAY
       start_date  from_station_id  trips   Latitude  Longitude  dist_to_c_mi
554332 2017-09-10              373    2.0  41.895465 -87.706128      0.779204
554441 2017-09-10              508    2.0  41.909657 -87.716632      0.689346
554443 2017-09-10              510    2.0  41.902707 -87.709220      0.585091
561390 2017-09-24              373    2.0  41.895465 -87.706128      0.779204
561499 2017-09-24              508    1.0  41.909657 -87.716632      0.689346
561501 2017-09-24              510    4.0  41.902707 -87.709220      0.585091
568861 2017-10-09              373    1.0  41.895465 -87.706128      0.779204
568975 2017-10-09              508    5.0  41.909657 -87.716632      0.689346
568977 2017-10-09              510    6.0  41.902707 -87.709220      0.585091
575085 2017-10-22              373    2.0  41.895465 -87.706128      0.779204
575167 2017-10-22              508    2.0  41.909657 -87.716632      0.689346
575169 2017-10-22              510    3.0  41.902707 -8

In [99]:
# ============================================================
# 3. Radius Sensitivity: 0.5, 1.0, 1.5 miles
# ============================================================

soldiers_coords = (41.8625, -87.6167)
jackson_coords = (41.7831, -87.5819)

radius_results = []

for r in [0.5, 1.0, 1.5]:
    s_r = div_filter(divvy, soldiers_coords, r=r)
    j_r = div_filter(divvy, jackson_coords, r=r)

    s_gd_r, s_notgd_r = split_df_on_game_days(s_r, gds_by_yr)
    j_gd_r, j_notgd_r = split_df_on_game_days(j_r, gds_by_yr)

    s_gd_r = s_gd_r.copy()
    s_notgd_r = s_notgd_r.copy()
    j_gd_r = j_gd_r.copy()
    j_notgd_r = j_notgd_r.copy()

    s_gd_r["treated"] = 1
    s_gd_r["game_day"] = 1

    s_notgd_r["treated"] = 1
    s_notgd_r["game_day"] = 0

    j_gd_r["treated"] = 0
    j_gd_r["game_day"] = 1

    j_notgd_r["treated"] = 0
    j_notgd_r["game_day"] = 0

    df_r = pd.concat(
        [s_gd_r, s_notgd_r, j_gd_r, j_notgd_r],
        ignore_index=True
    )

    df_r = add_time_vars(df_r)
    df_r = df_r[df_r["year"].isin([2017, 2018])].copy()

    model_r = smf.ols(
        "trips ~ game_day + DiD + C(from_station_id) + C(dow) + C(month)",
        data=df_r
    ).fit(cov_type="HC3")

    radius_results.append({
        "radius_miles": r,
        "DiD_coef": model_r.params["DiD"],
        "p_value": model_r.pvalues["DiD"],
        "n_obs": int(model_r.nobs),
        "r_squared": model_r.rsquared
    })

radius_summary = pd.DataFrame(radius_results)
radius_summary

GAME DAY
       start_date  from_station_id  trips   Latitude  Longitude  dist_to_c_mi
553998 2017-09-10                3  247.0  41.867226 -87.615355      0.333785
553999 2017-09-10                4  130.0  41.856268 -87.613348      0.463860
554061 2017-09-10               72   20.0  41.860384 -87.625813      0.491205
554081 2017-09-10               97  185.0  41.865312 -87.617867      0.203361
554146 2017-09-10              168   31.0  41.864059 -87.623727      0.377293
...           ...              ...    ...        ...        ...           ...
768531 2018-12-16               72   13.0  41.860384 -87.625813      0.491205
768551 2018-12-16               97   32.0  41.865312 -87.617867      0.203361
768614 2018-12-16              168   12.0  41.864059 -87.623727      0.377293
768693 2018-12-16              255   38.0  41.867888 -87.623041      0.495025
768768 2018-12-16              338   12.0  41.857611 -87.619407      0.365396

[105 rows x 6 columns]
*NOT* GAME DAY
       start_dat

,radius_miles,DiD_coef,p_value,n_obs,r_squared
0,0.5,19.757937,0.000085,1565,0.477663
1,1.0,7.742029,0.000144,5045,0.507005
2,1.5,1.368757,0.319430,11562,0.508176


In [108]:
# ============================================================
# Revised Placebo Test: Same Seasons, Fake Nearby Non-Game Dates
# ============================================================

# Shift each actual game date forward by 2 days.
# This preserves seasonality/month/weather-ish timing,
# but should not capture actual Bears game traffic.
fake_gds = {d + pd.Timedelta(days=2) for d in all_gds}

# Make sure no fake date accidentally equals a real game date
fake_gds = fake_gds - all_gds

s_placebo = div_filter(divvy, soldiers_coords, r=1.0)
j_placebo = div_filter(divvy, jackson_coords, r=1.0)

fake_gds_by_yr = {
    "2017_2018_placebo": fake_gds
}

s_gd_p, s_notgd_p = split_df_on_game_days(s_placebo, fake_gds_by_yr)
j_gd_p, j_notgd_p = split_df_on_game_days(j_placebo, fake_gds_by_yr)

s_gd_p = s_gd_p.copy()
s_notgd_p = s_notgd_p.copy()
j_gd_p = j_gd_p.copy()
j_notgd_p = j_notgd_p.copy()

s_gd_p["treated"] = 1
s_gd_p["game_day"] = 1

s_notgd_p["treated"] = 1
s_notgd_p["game_day"] = 0

j_gd_p["treated"] = 0
j_gd_p["game_day"] = 1

j_notgd_p["treated"] = 0
j_notgd_p["game_day"] = 0

placebo_df = pd.concat(
    [s_gd_p, s_notgd_p, j_gd_p, j_notgd_p],
    ignore_index=True
)

placebo_df = add_time_vars(placebo_df)
placebo_df = placebo_df[placebo_df["year"].isin([2017, 2018])].copy()

placebo_model = smf.ols(
    "trips ~ game_day + DiD + C(from_station_id) + C(dow) + C(month)",
    data=placebo_df
).fit(cov_type="HC3")

print("Revised Placebo Test: Fake Nearby Non-Game Dates")
print(f"DiD coefficient: {placebo_model.params['DiD']:.3f}")
print(f"p-value:         {placebo_model.pvalues['DiD']:.3f}")

GAME DAY
       start_date  from_station_id  trips   Latitude  Longitude  dist_to_c_mi
554996 2017-09-12                2   36.0  41.876511 -87.620548      0.988131
554997 2017-09-12                3  118.0  41.867226 -87.615355      0.333785
554998 2017-09-12                4   90.0  41.856268 -87.613348      0.463860
554999 2017-09-12                5   26.0  41.874053 -87.627716      0.979012
555033 2017-09-12               41   61.0  41.872078 -87.629544      0.935233
...           ...              ...    ...        ...        ...           ...
769697 2018-12-18              341   14.0  41.866095 -87.607267      0.545252
769713 2018-12-18              370    4.0  41.854184 -87.619154      0.588281
769723 2018-12-18              394    9.0  41.870816 -87.631246      0.943576
769842 2018-12-18              623    8.0  41.872773 -87.623981      0.802603
769845 2018-12-18              626   13.0  41.867491 -87.632190      0.868451

[314 rows x 6 columns]
*NOT* GAME DAY
       start_dat

In [103]:
# ============================================================
# 5. Year-by-Year Estimates
# ============================================================

year_results = []

for yr in [2017, 2018]:
    df_y = df[df["year"] == yr].copy()

    model_y = smf.ols(
        "trips ~ game_day + DiD + C(from_station_id) + C(dow) + C(month)",
        data=df_y
    ).fit(cov_type="HC3")

    year_results.append({
        "year": yr,
        "DiD_coef": model_y.params["DiD"],
        "p_value": model_y.pvalues["DiD"],
        "n_obs": int(model_y.nobs),
        "r_squared": model_y.rsquared
    })

year_summary = pd.DataFrame(year_results)
year_summary

,year,DiD_coef,p_value,n_obs,r_squared
0,2017,9.286403,0.007079,2434,0.533603
1,2018,7.307435,0.000703,2611,0.574269


## Preliminary Findings

Our results suggest that Chicago Bears home games caused a meaningful increase in Divvy ridership near Soldier Field. Using a fixed-effects difference-in-differences specification comparing stations within 1 mile of Soldier Field to stations near Jackson Park, we estimate a treatment effect of approximately **7.74 additional trips per station-day** on Bears home game days. This estimate is statistically significant at the 1% level (`p < 0.01`).

The findings remain highly consistent when using Humboldt Park as an alternative control neighborhood. In that specification, the estimated treatment effect is approximately **7.59 additional trips per station-day** (`p < 0.01`), suggesting that our results are robust to the choice of comparison area.

Our radius sensitivity analysis also supports the interpretation that the effect is localized around Soldier Field. When restricting the treatment radius to **0.5 miles**, the estimated effect increases substantially to approximately **19.76 additional trips per station-day**. However, when expanding the radius to **1.5 miles**, the estimated effect falls to approximately **1.37 trips** and becomes statistically insignificant. This spatial decay pattern indicates that the increase in Divvy usage is concentrated closest to the stadium.

The year-by-year estimates are also consistent across seasons. For 2017, we estimate an effect of approximately **9.29 additional trips per station-day**, while the 2018 estimate is approximately **7.31 additional trips**, with both estimates statistically significant. This suggests that the observed relationship is not driven by a single season.

Finally, we implemented a revised placebo test by shifting actual Bears game dates forward several days while preserving seasonal timing. This placebo framework helps isolate the causal effect of game-day activity from broader seasonal or weekend cycling patterns. The revised placebo design provides additional support for the validity of our identification strategy.